# 01. LLM 리뷰 분석용 순수 전처리 코드

이 코드는 **LLM을 호출하기 전 단계의 순수 데이터 전처리**를 담당

## 역할
1. Steam 리뷰 데이터, 게임 메타데이터, 리뷰 히스토그램 데이터를 불러온다.
2. 각 데이터 테이블을 전처리 한다.
    - 필요한 컬럼을 정리한다.
    - 리뷰 작성일과 게임 출시일을 기준으로 출시 후 경과일을 계산한다.
    - LLM 분석 후보 리뷰에 필요한 파생 컬럼을 만든다.
3. 다음 LLM 실행 코드에서 사용할 전처리 완료 데이터를 저장한다.

## 이 코드에서 하지 않는 것
- 장르 필터링
- 분석 기간 필터링
- 게임당 리뷰 수 설정
- 리뷰 샘플링
- LLM 호출
- Vertex AI 설정
- LLM 프롬프트/스키마 설정

위 항목들은 이후 LLM 실행 코드에서 목적에 맞게 별도로 처리한다.


# 0. 데이터 테이블별 전처리 작업 요약

이 코드에서는 세 개의 원천 테이블을 불러와 LLM 분석 후보 데이터로 결합한다.

| 데이터 테이블 | 주요 역할 | 전처리 내용 | 최종 사용 방식 |
|---|---|---|---|
| `steam_indie_reviews.csv` | 개별 Steam 리뷰 데이터 | 리뷰 ID, 게임 ID, 작성일, 추천/비추천 라벨, 리뷰 본문, 플레이타임, 리뷰 품질 컬럼 정리 | LLM 분석 후보 리뷰의 기본 단위 |
| `steam_indie_games.csv` | 게임 메타데이터 | 게임명, 출시일, 장르, 카테고리, Steam 태그, 가격대 정리 | 리뷰에 게임 속성 정보 결합 |
| `steam_indie_review_histogram.csv` | 날짜별 리뷰 히스토그램 데이터 | 날짜형 변환, 긍정/부정/전체 리뷰 수 숫자형 변환, 게임별 요약 생성 | 후보 게임의 리뷰 규모 참고용 |
| 결합 데이터 `df_base` | 리뷰 + 게임 메타 + 히스토그램 요약 | 출시 후 경과일, 출시 기준 구간, 장르/카테고리/태그 표시용 텍스트 생성 | 최종 전처리 후보 데이터 생성 |

## 산출물

| 산출물 | 설명 |
|---|---|
| `llm_preprocessed_reviews.csv` | LLM 실행 파일에서 필터링/샘플링하기 전의 전처리 완료 후보 데이터 |
| `llm_candidate_game_summary.csv` | 게임별 후보 리뷰 수 요약 |



# 0. 환경 설정

## 0-1. 기본 라이브러리

In [1]:
# ============================================================
# 0-1. 기본 라이브러리 불러오기
# ============================================================
import os
import ast
import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# 데이터프레임을 확인할 때 컬럼과 긴 텍스트가 잘 보이도록 설정한다.
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

## 0-2. 프로젝트 경로 및 산출물 경로 설정

In [2]:
# ============================================================
# 프로젝트 경로 및 산출물 경로 설정
# ============================================================
ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로,
# data/preprocessed 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# 소스 파일이 들어있는 폴더
DATA_DIR = ROOT / "data" / "preprocessed"

# 결과 저장 폴더
RUN_NAME = "preprocess_llm"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 입력 파일명
REVIEW_FILE = "steam_indie_reviews.csv"
GAME_META_FILE = "steam_indie_games.csv"
REVIEW_HISTOGRAM_FILE = "steam_indie_review_histogram.csv"

# 입력 파일 경로
INPUT_REVIEW_PATH = DATA_DIR / REVIEW_FILE
GAME_META_PATH = DATA_DIR / GAME_META_FILE
REVIEW_HISTOGRAM_PATH = DATA_DIR / REVIEW_HISTOGRAM_FILE

# 순수 전처리 산출물 경로
# 이후 LLM 실행 코드에서 이 파일을 다시 읽어 필터링/샘플링을 진행한다.
PREPROCESSED_REVIEWS_PATH = OUTPUT_DIR / "llm_preprocessed_reviews.csv"
CANDIDATE_GAME_SUMMARY_PATH = OUTPUT_DIR / "llm_candidate_game_summary.csv"

print("프로젝트 루트:", ROOT)
print("입력 리뷰 파일:", INPUT_REVIEW_PATH)
print("전처리 후보 데이터 저장 경로:", PREPROCESSED_REVIEWS_PATH)

프로젝트 루트: c:\Users\joon5\Documents\github\steam-indie-game-analysis
입력 리뷰 파일: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\preprocessed\steam_indie_reviews.csv
전처리 후보 데이터 저장 경로: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\preprocess_llm\llm_preprocessed_reviews.csv


# 1. 전처리 기준 설정

In [3]:
# ============================================================
# 1. 전처리 점검 설정
# ============================================================
# RUN_CHECK_CELLS
# - True이면 전처리 결과 분포와 샘플 데이터를 화면에 출력한다.
# - False이면 점검 출력 없이 다음 단계로 넘어간다.

RUN_CHECK_CELLS = True

# 리뷰 품질 판정용 파생 컬럼 생성 기준
# 이 값은 LLM 실행 대상을 바로 필터링하는 설정이 아닌,
# 전처리 후보 데이터에 is_meaningful_review, meaningless_reason 컬럼을 만들기 위한 기준
# 실제로 의미 있는 리뷰만 사용할지 여부는 LLM 실행 파일에서 결정
MIN_MEANINGFUL_REVIEW_LEN = 60
MIN_ALPHA_WORDS = 5

print("RUN_CHECK_CELLS:", RUN_CHECK_CELLS)
print("MIN_MEANINGFUL_REVIEW_LEN:", MIN_MEANINGFUL_REVIEW_LEN)
print("MIN_ALPHA_WORDS:", MIN_ALPHA_WORDS)


RUN_CHECK_CELLS: True
MIN_MEANINGFUL_REVIEW_LEN: 60
MIN_ALPHA_WORDS: 5


# 2. 공통 함수

## 2-1. 타입 변환, 리스트 파싱, 구간 생성 함수

In [4]:
# pandas/numpy 타입은 JSON 저장 시 오류가 날 수 있으므로
# 기본 Python 타입으로 안전하게 바꾸기 위한 함수
def to_serializable(obj):
    """JSON 저장이 어려운 pandas/numpy 타입을 기본 Python 타입으로 변환"""
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

# 리스트/태그 파싱 함수
# CSV 안에 문자열로 저장된 리스트, 콤마 구분 문자열, 태그 JSON을 분석과 프롬프트에 쓰기 쉬운 Python 리스트로 바꾼다.
def parse_list_like(value):
    """문자열로 저장된 리스트/콤마 구분 값을 Python 리스트로 변환"""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []

    text = str(value).strip()
    if text == "":
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    return [x.strip() for x in text.split(",") if x.strip()]


def parse_tags(value):
    """Steam 태그 JSON/문자열을 리스트로 변환"""
    if isinstance(value, dict):
        return list(value.keys())
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if pd.isna(value):
        return []

    text = str(value).strip()
    if text == "":
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, dict):
            try:
                return [k for k, _ in sorted(parsed.items(), key=lambda x: x[1], reverse=True)]
            except Exception:
                return list(parsed.keys())
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    return [x.strip() for x in text.split(",") if x.strip()]


# 표시용 문자열/라벨 변환 함수
# 리스트 컬럼은 그대로 프롬프트에 넣기 어렵기 때문에 사람이 읽기 쉬운 문자열로 변환한다.
def list_to_text(values):
    """리스트를 보고서/프롬프트용 문자열로 변환"""
    if not isinstance(values, list):
        return ""
    return ", ".join([str(x) for x in values if str(x).strip()])


def bool_to_label(value):
    """Steam 추천 여부를 positive/negative로 변환"""
    if pd.isna(value):
        return "unknown"

    if isinstance(value, (bool, np.bool_)):
        return "positive" if value else "negative"

    text = str(value).lower().strip()

    if text in ["true", "1", "yes", "positive"]:
        return "positive"
    if text in ["false", "0", "no", "negative"]:
        return "negative"

    return "unknown"

# 출시일 기준 구간 생성 함수
# 리뷰 작성일이 출시일로부터 얼마나 지났는지에 따라 D0-D7, D8-D30, D0-D30 같은 분석 구간을 만든다.
def get_release_period_detail(days):
    """출시일 기준 세부 구간을 생성"""
    if pd.isna(days):
        return "unknown"
    if days < 0:
        return "pre_release"
    if days <= 7:
        return "D0-D7"
    if days <= 30:
        return "D8-D30"
    if days <= 90:
        return "D31-D90"
    if days <= 180:
        return "D91-D180"
    return "D181+"


def get_release_period_group(days):
    """출시일 기준 큰 구간을 생성한다. 본 분석에서는 D0-D30을 출시 초기 한 달 반응으로 사용한다."""
    if pd.isna(days):
        return "unknown"
    if days < 0:
        return "pre_release"
    if days <= 30:
        return "D0-D30"
    if days <= 90:
        return "D31-D90"
    if days <= 180:
        return "D91-D180"
    return "D181+"

# 리뷰 품질 참고 컬럼 생성 함수
def judge_review_meaningfulness(text):
    """
    리뷰가 LLM 분석에 의미 있는지 판단하기 위한 1차 규칙 기반 함수.
    이 함수는 필터링을 바로 수행하지 않고, 품질 참고 컬럼을 만들기 위해 사용한다.
    """
    if pd.isna(text):
        return False, "empty"

    text = str(text).strip()
    text_lower = text.lower()

    if len(text) < MIN_MEANINGFUL_REVIEW_LEN:
        return False, "too_short"

    words = re.findall(r"[a-zA-Z]{2,}", text_lower)
    if len(words) < MIN_ALPHA_WORDS:
        return False, "too_few_words"

    if re.search(r"(.)\1{8,}", text_lower):
        return False, "repeated_character"

    trivial_patterns = [
        r"^good game\.?$",
        r"^bad game\.?$",
        r"^nice game\.?$",
        r"^fun game\.?$",
        r"^10/10\.?$",
        r"^recommended\.?$",
        r"^not recommended\.?$",
        r"^trash\.?$",
        r"^boring\.?$",
    ]
    if any(re.match(pattern, text_lower) for pattern in trivial_patterns):
        return False, "trivial_phrase"

    return True, "meaningful"

# 플레이타임 구간 생성 함수
# 리뷰 작성 시점 플레이타임을 very_early/early/mid/late로 나눠 유저가 어느 정도 플레이한 뒤 리뷰를 작성했는지 참고
def get_playtime_stage(hours):
    """
    리뷰 작성 시점 플레이타임을 구간으로 나눈다.

    기준:
    - 30분 미만: very_early
    - 30분 이상 2시간 미만: early
    - 2시간 이상 10시간 미만: mid
    - 10시간 이상: late
    """
    if pd.isna(hours):
        return "unknown"
    if hours < 0.5:
        return "very_early"
    if hours < 2:
        return "early"
    if hours < 10:
        return "mid"
    return "late"

# 3. 데이터 로드

## 3-1. 원천 데이터 불러오기

In [5]:
# ============================================================
# 원천 데이터 로드
# ============================================================
# 세 개의 원천 테이블을 불러온다.
# 이 단계에서는 아직 필터링이나 샘플링을 하지 않는다.

df_reviews_raw = pd.read_csv(INPUT_REVIEW_PATH)
df_games_raw = pd.read_csv(GAME_META_PATH)
df_hist_raw = pd.read_csv(REVIEW_HISTOGRAM_PATH)

print("리뷰 데이터:", df_reviews_raw.shape)
print("게임 메타 데이터:", df_games_raw.shape)
print("리뷰 히스토리 데이터:", df_hist_raw.shape)

리뷰 데이터: (160644, 24)
게임 메타 데이터: (8730, 20)
리뷰 히스토리 데이터: (468589, 10)



# 4. 리뷰 데이터 전처리

## 4-1. `steam_indie_reviews.csv`에서 처리하는 내용

| 작업 | 설명 |
|---|---|
| 키 컬럼 정리 | `recommendationid`, `appid`를 병합과 식별에 적합한 타입으로 변환 |
| 날짜 컬럼 정리 | `created_date` 또는 `timestamp_created`를 사용해 `review_datetime` 생성 |
| Steam 라벨 생성 | `voted_up`을 `positive` / `negative` 형태의 `steam_label_text`로 변환 |
| 리뷰 텍스트 정리 | 결측 리뷰를 빈 문자열로 바꾸고 앞뒤 공백 제거 |
| 리뷰 길이 계산 | `review_len` 생성 |
| 리뷰 품질 판정 | `is_meaningful_review`, `meaningless_reason` 생성 |
| 플레이타임 정리 | 시간 단위 플레이타임 컬럼 생성 및 숫자형 변환 |
| 플레이타임 구간 생성 | `playtime_stage` 생성 |
| 필터링 조건 컬럼 보정 | `steam_purchase`, `received_for_free`, `written_during_early_access` 컬럼 존재 보장 |

이 단계에서는 리뷰를 삭제하지 않는다.  
필터링 여부는 다음 LLM 실행 코드에서 결정한다.


In [6]:
df_reviews = df_reviews_raw.copy()

# 키 컬럼 정리
df_reviews["recommendationid"] = df_reviews["recommendationid"].astype(str)
df_reviews["appid"] = pd.to_numeric(df_reviews["appid"], errors="coerce").astype("Int64")

# 날짜 컬럼 정리
# created_date가 있으면 우선 사용하고,
# 없거나 변환 실패한 경우 timestamp_created를 Unix timestamp로 변환한다.
if "created_date" in df_reviews.columns:
    df_reviews["review_datetime"] = pd.to_datetime(df_reviews["created_date"], errors="coerce")
else:
    df_reviews["review_datetime"] = pd.NaT

if "timestamp_created" in df_reviews.columns:
    fallback_created = pd.to_datetime(df_reviews["timestamp_created"], unit="s", errors="coerce")
    df_reviews["review_datetime"] = df_reviews["review_datetime"].fillna(fallback_created)

# 추천/비추천 라벨 정리
# voted_up=True  -> positive
# voted_up=False -> negative
# 이 라벨은 Steam 원본 라벨이고, LLM이 판단하는 감정과는 별개다.
if "voted_up" in df_reviews.columns:
    df_reviews["steam_label_text"] = df_reviews["voted_up"].apply(bool_to_label)
else:
    df_reviews["steam_label_text"] = "unknown"

# 리뷰 본문 정리
if "review" not in df_reviews.columns:
    df_reviews["review"] = ""

df_reviews["review_text_clean"] = df_reviews["review"].fillna("").astype(str).str.strip()
df_reviews["review_len"] = df_reviews["review_text_clean"].str.len()

# 리뷰 품질 참고 컬럼 생성
# 너무 짧거나 의미가 약한 리뷰를 표시하기 위한 참고 컬럼
meaning_result = df_reviews["review_text_clean"].apply(judge_review_meaningfulness)
df_reviews["is_meaningful_review"] = meaning_result.apply(lambda x: x[0])
df_reviews["meaningless_reason"] = meaning_result.apply(lambda x: x[1])

# 플레이타임/투표 컬럼 정리
# 플레이타임, 투표 수, 가중 점수는 이후 프롬프트 맥락과 보고서 해석에 사용한다.
# 주의:
# 현재 전처리 데이터에는 playtime_at_review_hours, playtime_forever_hours가 이미 시간 단위로 존재할 수 있다.
# 기존 코드처럼 playtime_at_review 컬럼만 확인하면, 이미 존재하는 *_hours 값을 NaN으로 덮어쓸 수 있다.
# 따라서 *_hours 컬럼이 있으면 우선 사용하고, 분 단위 원본 컬럼이 있을 때만 /60 변환한다.
for col in [
    "playtime_at_review_hours",
    "playtime_forever_hours",
    "playtime_at_review",
    "playtime_forever",
    "author_playtime_at_review",
    "author_playtime_forever",
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
]:
    if col in df_reviews.columns:
        df_reviews[col] = pd.to_numeric(df_reviews[col], errors="coerce")

if "playtime_at_review_hours" in df_reviews.columns:
    df_reviews["playtime_at_review_hours"] = pd.to_numeric(
        df_reviews["playtime_at_review_hours"], errors="coerce"
    )
elif "playtime_at_review" in df_reviews.columns:
    df_reviews["playtime_at_review_hours"] = pd.to_numeric(
        df_reviews["playtime_at_review"], errors="coerce"
    ) / 60
elif "author_playtime_at_review" in df_reviews.columns:
    df_reviews["playtime_at_review_hours"] = pd.to_numeric(
        df_reviews["author_playtime_at_review"], errors="coerce"
    ) / 60
else:
    df_reviews["playtime_at_review_hours"] = np.nan

if "playtime_forever_hours" in df_reviews.columns:
    df_reviews["playtime_forever_hours"] = pd.to_numeric(
        df_reviews["playtime_forever_hours"], errors="coerce"
    )
elif "playtime_forever" in df_reviews.columns:
    df_reviews["playtime_forever_hours"] = pd.to_numeric(
        df_reviews["playtime_forever"], errors="coerce"
    ) / 60
elif "author_playtime_forever" in df_reviews.columns:
    df_reviews["playtime_forever_hours"] = pd.to_numeric(
        df_reviews["author_playtime_forever"], errors="coerce"
    ) / 60
else:
    df_reviews["playtime_forever_hours"] = np.nan

# 리뷰 작성 시점 플레이타임 구간 생성
# LLM 실행 파일의 프롬프트와 출시 후 보고서에서 참고할 수 있는 맥락 정보
df_reviews["playtime_stage"] = df_reviews["playtime_at_review_hours"].apply(get_playtime_stage)

# 필터링 조건용 bool 컬럼 보정
for col in ["steam_purchase", "received_for_free", "written_during_early_access"]:
    if col not in df_reviews.columns:
        df_reviews[col] = np.nan

print("리뷰 전처리 결과:", df_reviews.shape)


리뷰 전처리 결과: (160644, 31)



# 5. 게임 메타데이터 전처리

## 5-1. `steam_indie_games.csv`에서 처리하는 내용

| 작업 | 설명 |
|---|---|
| 게임 ID 정리 | `appid`를 숫자형으로 변환 |
| 게임명 생성 | `name`을 기준으로 `game_name` 생성 |
| 출시일 변환 | `release_date`를 날짜형으로 변환 |
| 장르 정리 | `genres`를 리스트형 `genres_list`로 변환 |
| 카테고리 정리 | `categories`를 리스트형 `categories_list`로 변환 |
| Steam 태그 정리 | `tags`를 리스트형 `steam_tags_list`로 변환 |
| 주요 태그 생성 | 상위 10개 태그를 `top_steam_tags_list`로 저장 |
| 가격대 생성 | `price` 또는 `price_spy`를 기준으로 `price_group` 생성 |

이 테이블은 리뷰 데이터와 `appid` 기준으로 결합된다.


In [7]:
df_games = df_games_raw.copy()

# 키/이름 컬럼 정리
df_games["appid"] = pd.to_numeric(df_games["appid"], errors="coerce").astype("Int64")

if "name" not in df_games.columns:
    df_games["name"] = df_games["appid"].astype(str)

df_games["game_name"] = df_games["name"].fillna(df_games["appid"].astype(str)).astype(str)

# 출시일 정리
if "release_date" in df_games.columns:
    df_games["release_date"] = pd.to_datetime(df_games["release_date"], errors="coerce")
else:
    df_games["release_date"] = pd.NaT

# 장르/카테고리/태그 정리
# genres, categories, tags는 문자열로 저장되어 있을 수 있으므로 리스트 형태로 변환해 필터링과 프롬프트 생성에 사용할 수 있게 만든다.
df_games["genres_list"] = (df_games["genres"].apply(parse_list_like) if "genres" in df_games.columns else [[] for _ in range(len(df_games))])
df_games["categories_list"] = (df_games["categories"].apply(parse_list_like) if "categories" in df_games.columns else [[] for _ in range(len(df_games))])
df_games["steam_tags_list"] = (df_games["tags"].apply(parse_tags) if "tags" in df_games.columns else [[] for _ in range(len(df_games))])
df_games["top_steam_tags_list"] = (df_games["steam_tags_list"].apply(lambda x: x[:10] if isinstance(x, list) else []))

# 가격 컬럼 정리 및 가격대 생성
# price_group은 현재 price 컬럼을 기준으로 만든다.
# 현재 steam_indie_games.csv의 price는 USD 단위로 해석한다.
# price_spy처럼 원 단위 가격을 사용할 경우 별도 환산 또는 다른 가격 구간 기준이 필요하다.
if "price" in df_games.columns:
    df_games["price"] = pd.to_numeric(df_games["price"], errors="coerce")
elif "price_spy" in df_games.columns:
    df_games["price"] = pd.to_numeric(df_games["price_spy"], errors="coerce")
else:
    df_games["price"] = np.nan

def make_price_group(price):
    if pd.isna(price):
        return "unknown"
    if price <= 0:
        return "free"
    if price < 5:
        return "0-5"
    if price < 10:
        return "5-10"
    if price < 20:
        return "10-20"
    if price < 40:
        return "20-40"
    return "40+"

df_games["price_group"] = df_games["price"].apply(make_price_group)

# 조인에 필요한 컬럼만 추림
# 리뷰 데이터에 붙일 핵심 게임 메타 컬럼만 남긴다
df_games_meta = df_games[[
    "appid",
    "game_name",
    "release_date",
    "genres_list",
    "categories_list",
    "steam_tags_list",
    "top_steam_tags_list",
    "price",
    "price_group",
]].drop_duplicates("appid")

print("게임 메타 전처리 결과:", df_games_meta.shape)

게임 메타 전처리 결과: (8730, 9)



# 6. 리뷰 히스토그램 데이터 전처리

## 6-1. `steam_indie_review_histogram.csv`에서 처리하는 내용

| 작업 | 설명 |
|---|---|
| 게임 ID 정리 | `appid`를 숫자형으로 변환 |
| 날짜 컬럼 변환 | `release_date`, `hist_start_date`, `hist_end_date`, `date`를 날짜형으로 변환 |
| 리뷰 수 컬럼 변환 | 긍정/부정/전체 리뷰 수를 숫자형으로 변환 |
| 전체 리뷰 수 보정 | `recommendations_total`이 없으면 긍정 + 부정으로 생성 |
| 게임별 요약 | 게임별 전체/긍정/부정 리뷰 수, 최초/최신 날짜를 요약 |

현재 LLM 입력에는 히스토그램 상세 데이터가 직접 들어가지는 않는다.  
다만 후보 게임의 리뷰 규모를 확인할 수 있도록 게임 단위 요약을 결합한다.


In [8]:
df_hist = df_hist_raw.copy()

# 키/날짜 컬럼 정리
df_hist["appid"] = pd.to_numeric(df_hist["appid"], errors="coerce").astype("Int64")

for col in ["release_date", "hist_start_date", "hist_end_date", "date"]:
    if col in df_hist.columns:
        df_hist[col] = pd.to_datetime(df_hist[col], errors="coerce")

# 리뷰 수 숫자형 컬럼 정리
# # 긍정/부정/전체 리뷰 수는 합산해야 하므로 숫자형으로 변환
for col in ["recommendations_up", "recommendations_down", "recommendations_total"]:
    if col in df_hist.columns:
        df_hist[col] = pd.to_numeric(df_hist[col], errors="coerce").fillna(0)

if "recommendations_total" not in df_hist.columns and {"recommendations_up", "recommendations_down"}.issubset(df_hist.columns):
    df_hist["recommendations_total"] = df_hist["recommendations_up"] + df_hist["recommendations_down"]

# 게임별 히스토그램 요약
# 현재 LLM 입력에는 날짜별 히스토그램을 직접 넣지는 않지만, 후보 게임의 리뷰 규모를 참고할 수 있도록 게임 단위로 요약한다.
agg_dict = {}
if "recommendations_total" in df_hist.columns:
    agg_dict["hist_total_reviews"] = ("recommendations_total", "sum")
if "recommendations_up" in df_hist.columns:
    agg_dict["hist_positive_reviews"] = ("recommendations_up", "sum")
if "recommendations_down" in df_hist.columns:
    agg_dict["hist_negative_reviews"] = ("recommendations_down", "sum")
if "date" in df_hist.columns:
    agg_dict["hist_first_date"] = ("date", "min")
    agg_dict["hist_last_date"] = ("date", "max")

if agg_dict:
    df_hist_game_summary = df_hist.groupby("appid", as_index=False).agg(**agg_dict)
else:
    df_hist_game_summary = pd.DataFrame({"appid": df_hist["appid"].dropna().unique()})

print("히스토그램 게임 요약:", df_hist_game_summary.shape)


히스토그램 게임 요약: (8526, 6)



# 7. 데이터 결합 및 파생 컬럼 생성

## 7-1. 결합 후 만드는 주요 컬럼

| 컬럼 | 설명 |
|---|---|
| `genres_text` | 장르 리스트를 문자열로 변환한 값 |
| `categories_text` | 카테고리 리스트를 문자열로 변환한 값 |
| `top_steam_tags_text` | 주요 Steam 태그 리스트를 문자열로 변환한 값 |
| `days_from_release` | 게임 출시일부터 리뷰 작성일까지 지난 일수 |
| `release_period_detail` | 기존 세부 구간. 예: `D0-D7`, `D8-D30` |
| `release_period` | 본 분석용 큰 구간. 예: `D0-D30` |

`release_period`는 다음 LLM 실행 파일에서 출시 초기 한 달 리뷰를 필터링할 때 사용한다.


In [9]:
# 리뷰 + 게임 메타 결합
# 개별 리뷰에 게임명, 장르, 카테고리, 태그, 가격 정보를 붙인다.
before = len(df_reviews)
df_base = df_reviews.merge(df_games_meta, on="appid", how="left")

# 리뷰 + 히스토그램 요약 결합
# 후보 게임의 전체 리뷰 규모를 참고할 수 있도록 히스토그램 요약을 붙인다.
# 히스토그램 데이터는 후보 게임의 리뷰 규모를 참고하기 위해 중간 결합하지만, 현재 LLM 입력 저장 컬럼에는 포함하지 않는다.
before = len(df_base)
df_base = df_base.merge(df_hist_game_summary, on="appid", how="left")

# 결측 보정
# 게임명이 없으면 appid 문자열로 대체한다.
# 리스트 컬럼은 결측이 생기면 apply 단계에서 오류가 날 수 있으므로 빈 리스트로 보정한다.
df_base["game_name"] = df_base["game_name"].fillna(df_base["appid"].astype(str))

for col in ["genres_list", "categories_list", "steam_tags_list", "top_steam_tags_list"]:
    if col in df_base.columns:
        df_base[col] = df_base[col].apply(lambda x: x if isinstance(x, list) else [])

#  표시용 텍스트 컬럼 생성
# 리스트 컬럼을 프롬프트와 확인 표에서 보기 쉬운 문자열로 변환한다.
df_base["genres_text"] = df_base["genres_list"].apply(list_to_text)
df_base["categories_text"] = df_base["categories_list"].apply(list_to_text)
df_base["top_steam_tags_text"] = df_base["top_steam_tags_list"].apply(list_to_text)

# 출시일 기준 리뷰 작성 경과일 생성
# 출시일보다 리뷰 작성일이 빠르면 pre_release로 분류된다.
# 얼리액세스 시절 리뷰가 여기에 포함될 수 있다.
if "release_date" in df_base.columns:
    df_base["days_from_release"] = (
        df_base["review_datetime"].dt.normalize() - df_base["release_date"].dt.normalize()
    ).dt.days
else:
    df_base["days_from_release"] = np.nan

# 출시 기준 구간
# release_period_detail은 D0-D7, D8-D30 같은 세부 구간 확인용으로 유지한다.
# release_period는 본 분석용 큰 구간으로 D0-D30을 포함한다.
df_base["release_period_detail"] = df_base["days_from_release"].apply(get_release_period_detail)
df_base["release_period"] = df_base["days_from_release"].apply(get_release_period_group)

print("결합 데이터:", df_base.shape)


결합 데이터: (160644, 50)



# 8. 전처리 결과 점검

전처리 결과가 의도대로 만들어졌는지 확인한다.

확인 항목은 다음과 같다.

| 확인 항목 | 목적 |
|---|---|
| 게임명 결측 수 | 게임 메타 결합 여부 확인 |
| 출시일 결측 수 | 출시 후 경과일 계산 가능 여부 확인 |
| 리뷰 작성일 결측 수 | 리뷰 시점 분석 가능 여부 확인 |
| `release_period` 분포 | 출시 기준 구간 생성 결과 확인 |
| `is_meaningful_review` 분포 | 의미 있는 리뷰 후보 비율 확인 |
| `meaningless_reason` 분포 | 의미 없는 리뷰로 판단된 이유 확인 |


In [10]:
# 전처리 결과 점검
if RUN_CHECK_CELLS:
    print("게임명 결측 수:", df_base["game_name"].isna().sum())
    print("출시일 결측 수:", df_base["release_date"].isna().sum())
    print("review_datetime 결측 수:", df_base["review_datetime"].isna().sum())
    print("release_period 분포")
    display(df_base["release_period"].value_counts(dropna=False))
    print("review quality 분포")
    display(df_base["is_meaningful_review"].value_counts(dropna=False))
    print("meaningless reason 분포")
    display(df_base["meaningless_reason"].value_counts(dropna=False).head(20))
    display(df_base.head())


게임명 결측 수: 0
출시일 결측 수: 0
review_datetime 결측 수: 0
release_period 분포


release_period
D181+          61249
pre_release    40253
D0-D30         29130
D31-D90        15222
D91-D180       14790
Name: count, dtype: int64

review quality 분포


is_meaningful_review
False    106645
True      53999
Name: count, dtype: int64

meaningless reason 분포


meaningless_reason
too_short             85486
meaningful            53999
too_few_words         20525
repeated_character      634
Name: count, dtype: int64

,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author_steamid,author_num_games_owned,author_num_reviews,author_last_played,created_date,updated_date,author_last_played_date,playtime_forever_hours,playtime_last_two_weeks_hours,playtime_at_review_hours,review_datetime,steam_label_text,review_text_clean,review_len,is_meaningful_review,meaningless_reason,playtime_stage,game_name,release_date,genres_list,categories_list,steam_tags_list,top_steam_tags_list,price,price_group,hist_total_reviews,hist_positive_reviews,hist_negative_reviews,hist_first_date,hist_last_date,genres_text,categories_text,top_steam_tags_text,days_from_release,release_period_detail,release_period
0,18698790,324470,french,good game for this price,1445883033,1445883033,True,2,1,0.523810,0,True,False,True,76561197986244887,0,3,1503258537,2015-10-26 18:10:33,2015-10-26 18:10:33,2017-08-20 19:48:57,3.483333,0.0,1.250000,2015-10-26 18:10:33,positive,good game for this price,24,False,too_short,early,SinaRun,2025-11-03,"[Indie, Racing]","[Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, ...","[Indie, Racing, Early Access, Parkour, First-Person]","[Indie, Racing, Early Access, Parkour, First-Person]",3.99,0-5,NaN,NaN,NaN,NaT,NaT,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",-3661,pre_release,pre_release
1,18699465,324470,english,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effect is too excessive \nSo level of difficulty is too hard for beginner ...",1445885616,1445885616,True,1,0,0.421372,0,True,False,True,76561198049920411,0,3,1445958836,2015-10-26 18:53:36,2015-10-26 18:53:36,2015-10-27 15:13:56,0.216667,0.0,0.216667,2015-10-26 18:53:36,positive,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effect is too excessive \nSo level of difficulty is too hard for beginner ...",119,True,meaningful,very_early,SinaRun,2025-11-03,"[Indie, Racing]","[Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, ...","[Indie, Racing, Early Access, Parkour, First-Person]","[Indie, Racing, Early Access, Parkour, First-Person]",3.99,0-5,NaN,NaN,NaN,NaT,NaT,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",-3661,pre_release,pre_release
2,18699648,324470,english,"this game is like a zen-garden, I love it! \n\npros:\n-it's very relaxing\n-good controls\n-awesome leveldesign\n-re...",1445886217,1482541338,True,5,0,0.500076,1,True,False,True,76561198047893068,435,17,1623275518,2015-10-26 19:03:37,2016-12-24 01:02:18,2021-06-09 21:51:58,13.616667,0.0,12.666667,2015-10-26 19:03:37,positive,"this game is like a zen-garden, I love it! \n\npros:\n-it's very relaxing\n-good controls\n-awesome leveldesign\n-re...",387,True,meaningful,late,SinaRun,2025-11-03,"[Indie, Racing]","[Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, ...","[Indie, Racing, Early Access, Parkour, First-Person]","[Indie, Racing, Early Access, Parkour, First-Person]",3.99,0-5,NaN,NaN,NaN,NaT,NaT,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",-3661,pre_release,pre_release
3,18700348,324470,english,Ever played Bhop? Surf? If so this games mechanics will feel Instantly similar too you. This game gives you such a r...,1445889159,1445895001,True,16,0,0.637511,1,True,False,True,76561198045694190,0,3,1541537929,2015-10-26 19:52:39,2015-10-26 21:30:01,2018-11-06 20:58:49,7.483333,0.0,0


# 9. 저장 대상 데이터 구성

이 단계에서는 저장할 컬럼을 정하고, 저장용 데이터프레임을 만든다.

저장 코드는 다음 섹션인 **10. 전처리 산출물 저장**에 따로 분리한다.


In [11]:
# ============================================================
# 9. 저장 대상 컬럼 구성 및 저장용 데이터프레임 생성
# ============================================================
# 이 셀에서는 저장할 컬럼 목록을 정하고, 저장 직전의 데이터프레임을 만든다.
# 실제 to_csv 저장 코드는 다음 셀에서만 수행한다.

# LLM 실행 파일에서 필터링/샘플링하기 위한 후보 데이터 컬럼
# 분석 조건 필터링에 필요한 컬럼
# LLM 프롬프트에 들어갈 수 있는 리뷰/게임 정보
# 리뷰 품질 판단에 필요한 컬럼만 남긴다.
PREPROCESSED_COLUMNS = [
    "recommendationid",
    "appid",
    "game_name",
    "language",
    "review_datetime",
    "release_date",
    "days_from_release",
    "release_period",
    "release_period_detail",
    "steam_label_text",
    "voted_up",
    "playtime_at_review_hours",
    "playtime_forever_hours",
    "playtime_stage",
    "votes_up",
    "weighted_vote_score",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "genres_text",
    "categories_text",
    "top_steam_tags_text",
    "price",
    "price_group",
    "review_text_clean",
    "review_len",
    "is_meaningful_review",
    "meaningless_reason",
    "hist_total_reviews",
    "hist_positive_reviews",
    "hist_negative_reviews",
    "hist_first_date",
    "hist_last_date",
]

# 실제 df_base에 존재하는 컬럼만 선택한다.
# 데이터 버전에 따라 일부 컬럼이 없더라도 에러가 나지 않게 하기 위한 안전장치다.
preprocessed_cols = [c for c in PREPROCESSED_COLUMNS if c in df_base.columns]
df_preprocessed = df_base[preprocessed_cols].copy()

# 게임별 후보 리뷰 수 요약 생성
# LLM 실행 전, 어떤 게임에 후보 리뷰가 얼마나 있는지 확인하기 위한 참고 산출물이다.
candidate_game_summary = (
    df_preprocessed
    .groupby(["appid", "game_name"], as_index=False)
    .agg(
        candidate_review_count=("recommendationid", "count"),
        meaningful_review_count=("is_meaningful_review", "sum"),
        positive_count=("steam_label_text", lambda x: (x == "positive").sum()),
        negative_count=("steam_label_text", lambda x: (x == "negative").sum()),
        first_review_datetime=("review_datetime", "min"),
        last_review_datetime=("review_datetime", "max"),
    )
    .sort_values("candidate_review_count", ascending=False)
)

print("저장 예정 전처리 후보 데이터:", df_preprocessed.shape)
print("저장 예정 게임별 후보 요약:", candidate_game_summary.shape)


저장 예정 전처리 후보 데이터: (160644, 33)
저장 예정 게임별 후보 요약: (192, 8)


# 10. 전처리 산출물 저장

In [12]:
# ============================================================
# 10. 전처리 산출물 저장
# ============================================================
# 데이터 저장 코드는 이 셀에만 모아둔다.
# 저장 파일명이나 저장 경로를 바꾸고 싶을 때는 이 셀과 경로 설정 셀을 확인하면 된다.

df_preprocessed.to_csv(PREPROCESSED_REVIEWS_PATH, index=False, encoding="utf-8-sig")
candidate_game_summary.to_csv(CANDIDATE_GAME_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("전처리 후보 데이터 저장:", PREPROCESSED_REVIEWS_PATH, df_preprocessed.shape)
print("게임별 후보 요약 저장:", CANDIDATE_GAME_SUMMARY_PATH, candidate_game_summary.shape)

# 저장된 데이터의 앞부분을 확인한다.
display(df_preprocessed.head())
display(candidate_game_summary.head())


전처리 후보 데이터 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\preprocess_llm\llm_preprocessed_reviews.csv (160644, 33)
게임별 후보 요약 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\preprocess_llm\llm_candidate_game_summary.csv (192, 8)


,recommendationid,appid,game_name,language,review_datetime,release_date,days_from_release,release_period,release_period_detail,steam_label_text,voted_up,playtime_at_review_hours,playtime_forever_hours,playtime_stage,votes_up,weighted_vote_score,steam_purchase,received_for_free,written_during_early_access,genres_text,categories_text,top_steam_tags_text,price,price_group,review_text_clean,review_len,is_meaningful_review,meaningless_reason,hist_total_reviews,hist_positive_reviews,hist_negative_reviews,hist_first_date,hist_last_date
0,18698790,324470,SinaRun,french,2015-10-26 18:10:33,2025-11-03,-3661,pre_release,pre_release,positive,True,1.250000,3.483333,early,2,0.523810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,good game for this price,24,False,too_short,NaN,NaN,NaN,NaT,NaT
1,18699465,324470,SinaRun,english,2015-10-26 18:53:36,2025-11-03,-3661,pre_release,pre_release,positive,True,0.216667,0.216667,very_early,1,0.421372,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effect is too excessive \nSo level of difficulty is too hard for beginner ...",119,True,meaningful,NaN,NaN,NaN,NaT,NaT
2,18699648,324470,SinaRun,english,2015-10-26 19:03:37,2025-11-03,-3661,pre_release,pre_release,positive,True,12.666667,13.616667,late,5,0.500076,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"this game is like a zen-garden, I love it! \n\npros:\n-it's very relaxing\n-good controls\n-awesome leveldesign\n-re...",387,True,meaningful,NaN,NaN,NaN,NaT,NaT
3,18700348,324470,SinaRun,english,2015-10-26 19:52:39,2025-11-03,-3661,pre_release,pre_release,positive,True,0.916667,7.483333,early,16,0.637511,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,Ever played Bhop? Surf? If so this games mechanics will feel Instantly similar too you. This game gives you such a r...,1100,True,meaningful,NaN,NaN,NaN,NaT,NaT
4,18701774,324470,SinaRun,english,2015-10-26 21:32:24,2025-11-03,-3661,pre_release,pre_release,positive,True,6.416667,8.666667,mid,4,0.495810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,It's Lit,8,False,too_short,NaN,NaN,NaN,NaT,NaT


,appid,game_name,candidate_review_count,meaningful_review_count,positive_count,negative_count,first_review_datetime,last_review_datetime
57,1931770,Chants of Sennaar,26883,11440,26440,443,2023-09-05 19:39:58,2026-05-02 17:39:04
12,1189490,觅长生,26357,400,24582,1775,2019-11-26 11:03:35,2026-05-01 04:50:38
1,571740,Golf It!,24233,6350,21979,2254,2017-02-17 15:44:06,2026-05-02 14:33:28
67,2114740,Blasphemous 2,13819,5951,12572,1247,2023-08-24 16:21:11,2026-05-01 09:27:00
4,619820,Heroes of Hammerwatch II,5435,2681,4518,917,2025-01-14 22:35:14,2026-04-28 15:48:01


# 11. 출력 테이블 설명
`llm_preprocessed_reviews.csv` 컬럼 명세
- 설명: LLM 분석 전에 사용할 리뷰 후보 데이터

| 컬럼명                           | 설명                            | 예시 값                                                                      |
| ----------------------------- | ----------------------------- | ------------------------------------------------------------------------- |
| `recommendationid`            | Steam 리뷰 고유 ID                | 144515690                                                                 |
| `appid`                       | Steam 게임 고유 ID                | 571740                                                                    |
| `game_name`                   | Steam 게임명                     | Golf It!                                                                  |
| `language`                    | 리뷰 작성 언어                      | english                                                                   |
| `review_datetime`             | 리뷰 작성 일시                      | 2023-08-18 21:03:00                                                       |
| `release_date`                | 게임 출시일                        | 2023-08-18                                                                |
| `days_from_release`           | 출시일 기준 리뷰 작성일까지 지난 일수         | 0                                                                         |
| `release_period`              | 출시 후 리뷰 작성 구간                 | D0-D30                                                                    |
| `release_period_detail`       | 출시 후 세부 리뷰 작성 구간              | D0-D7                                                                     |
| `steam_label_text`            | Steam 추천 여부를 문자열로 변환한 값       | positive                                                                  |
| `voted_up`                    | Steam 원본 추천 여부                | True                                                                      |
| `playtime_at_review_hours`    | 리뷰 작성 시점 플레이타임                | 2.9                                                                       |
| `playtime_forever_hours`      | 유저의 전체 누적 플레이타임               | 15.4                                                                      |
| `playtime_stage`              | 리뷰 작성 시점 플레이타임 구간             | 1-5h                                                                      |
| `votes_up`                    | 리뷰가 받은 유용함 투표 수               | 6                                                                         |
| `weighted_vote_score`         | Steam 리뷰 가중 점수                | 0.563                                                                     |
| `steam_purchase`              | Steam 직접 구매 여부                | True                                                                      |
| `received_for_free`           | 무료 수령 여부                      | False                                                                     |
| `written_during_early_access` | 얼리액세스 기간 작성 여부                | False                                                                     |
| `genres_text`                 | 게임의 Steam 장르 목록을 쉼표로 연결한 값    | Casual, Indie, Simulation, Sports                                         |
| `categories_text`             | Steam 카테고리 목록을 쉼표로 연결한 값      | Single-player, Multi-player, PvP                                          |
| `top_steam_tags_text`         | 게임의 주요 Steam 태그 목록을 쉼표로 연결한 값 | Multiplayer, Mini Golf, Golf, Casual                                      |
| `price`                       | 게임 가격. 현재 데이터 기준 USD 단위로 해석   | 8.99                                                                      |
| `price_group`                 | 게임 가격을 분석용 구간으로 나눈 값          | 5-10                                                                      |
| `review_text_clean`           | 정제된 리뷰 본문                     | I love the game and now that it is out of early access it is even better. |
| `review_len`                  | 정제된 리뷰 본문 길이                  | 78                                                                        |
| `is_meaningful_review`        | LLM 분석에 사용할 만큼 의미 있는 리뷰인지 여부  | True                                                                      |
| `meaningless_reason`          | 무의미한 리뷰로 판단된 이유               | meaningful                                                                |
| `hist_total_reviews`          | 리뷰 히스토그램 기준 해당 게임의 누적 리뷰 수    | 11480                                                                     |
| `hist_positive_reviews`       | 리뷰 히스토그램 기준 긍정 리뷰 수           | 10043                                                                     |
| `hist_negative_reviews`       | 리뷰 히스토그램 기준 부정 리뷰 수           | 1437                                                                      |
| `hist_first_date`             | 리뷰 히스토그램에서 확인된 가장 이른 날짜       | 2023-08-18                                                                |
| `hist_last_date`              | 리뷰 히스토그램에서 확인된 가장 최근 날짜       | 2026-04-26                                                                |


`llm_candidate_game_summary.csv` 컬럼 명세
- 설명: 게임별 LLM 후보 리뷰 수 요약 데이터

| 컬럼명                       | 설명                                                   | 예시 값                |
| ------------------------- | ---------------------------------------------------- | ------------------- |
| `appid`                   | Steam 게임 고유 ID                                       | 571740              |
| `game_name`               | Steam 게임명                                            | Golf It!            |
| `candidate_review_count`  | 전처리 후 해당 게임에 남아 있는 전체 후보 리뷰 수. 아직 LLM 실행 조건 필터링 전 기준 | 240                 |
| `meaningful_review_count` | 전처리 기준상 의미 있는 리뷰로 표시된 리뷰 수                           | 225                 |
| `positive_count`          | Steam 기준 긍정 리뷰 수                                     | 190                 |
| `negative_count`          | Steam 기준 부정 리뷰 수                                     | 50                  |
| `first_review_datetime`   | 해당 게임의 가장 이른 리뷰 작성 시점                                | 2023-08-18 21:03:00 |
| `last_review_datetime`    | 해당 게임의 가장 최근 리뷰 작성 시점                                | 2026-04-05 16:40:12 |

